# Data Audit

This notebook reviews the adjusted-close dataset produced by `make data`. It focuses on coverage, missing observations, and basic price and return visualizations. It does not perform modeling.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

project_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(project_root / "src"))

In [ ]:
from mrrp.data.cache import load_parquet
from mrrp.data.validators import report_missing_data

## 1. Load Adjusted Close Data

In [ ]:
data_path = project_root / "data/processed/adjusted_close.parquet"
prices = load_parquet(data_path)
prices.head()

## 2. Ticker Coverage

In [ ]:
coverage_summary = pd.Series(
    {
        "ticker_count": len(prices.columns),
        "start_date": prices.index.min(),
        "end_date": prices.index.max(),
        "row_count": len(prices),
    },
    name="dataset",
)
coverage_summary

In [ ]:
ticker_list = prices.columns.tolist()
ticker_list

In [ ]:
observation_counts = prices.count().rename("observation_count").to_frame()
observation_counts

## 3. Missing Data

In [ ]:
missing_data = report_missing_data(prices)
missing_data.sort_values("missing_fraction", ascending=False)

## 4. Normalized Prices

Each ticker is rebased to 100 at its own first valid observation so assets with different inception dates can be compared without filling backward.

In [ ]:
normalized_prices = prices.apply(lambda series: series / series.dropna().iloc[0] * 100)
normalized_figure = px.line(
    normalized_prices,
    title="Normalized Adjusted Close Prices",
    labels={"value": "Normalized Price", "Date": "Date", "variable": "Ticker"},
)
normalized_figure

## 5. Recent Daily Returns

The chart shows the most recent 252 observations for a few representative tickers. Returns are shown only as a data-quality view, not as a risk model.

In [ ]:
audit_tickers = [ticker for ticker in ["SPY", "QQQ", "XIU.TO"] if ticker in prices]
daily_returns = prices.pct_change(fill_method=None)
returns_figure = px.line(
    daily_returns[audit_tickers].tail(252),
    title="Recent Daily Returns",
    labels={"value": "Daily Return", "Date": "Date", "variable": "Ticker"},
)
returns_figure

## 6. Data Limitations

- The dataset uses free Yahoo Finance ETF data and is not institutional-grade.
- Tickers can have different inception dates, trading calendars, and missing observations.
- Adjusted-close history may be revised by the upstream provider.
- ETF prices are proxies and do not provide direct holdings-level, factor, or economic exposure data.
- This audit does not establish data fitness for investment decisions and is not financial advice.